In [ ]:
import numpy as np
import pandas as pd

# =====================================================
# Longstaff-Schwartz Method
# =====================================================

def price_american_put_lsm(
    S0=10,
    K=10,
    r=0.04,
    sigma=0.40,
    T=1/12,
    n_paths=50000,
    n_steps=252,
    seed=42
):

    np.random.seed(seed)

    dt = T / n_steps

    # -------------------------------------------------
    # Simulate stock price paths
    # -------------------------------------------------
    S = np.zeros((n_paths, n_steps + 1))
    S[:, 0] = S0

    for t in range(1, n_steps + 1):

        z = np.random.randn(n_paths)

        S[:, t] = S[:, t - 1] * np.exp(
            (r - 0.5 * sigma**2) * dt
            + sigma * np.sqrt(dt) * z
        )

    # -------------------------------------------------
    # Terminal payoff
    # -------------------------------------------------
    cashflows = np.maximum(K - S[:, -1], 0)

    exercise_time = np.full(n_paths, n_steps)

    # -------------------------------------------------
    # Backward induction
    # -------------------------------------------------
    for t in range(n_steps - 1, 0, -1):

        S_t = S[:, t]

        exercise_value = np.maximum(K - S_t, 0)

        # Only ITM paths
        itm = exercise_value > 0

        if np.sum(itm) == 0:
            continue

        X = S_t[itm]

        # Discount future cashflows back to current time
        Y = cashflows[itm] * np.exp(
            -r * dt * (exercise_time[itm] - t)
        )

        # -------------------------------------------------
        # Regression: 1, S, S^2
        # -------------------------------------------------
        coeff = np.polyfit(X, Y, deg=2)

        continuation_value = np.polyval(coeff, X)

        # -------------------------------------------------
        # Exercise decision
        # -------------------------------------------------
        exercise_now = exercise_value[itm] > continuation_value

        itm_indices = np.where(itm)[0]

        exercise_indices = itm_indices[exercise_now]

        cashflows[exercise_indices] = exercise_value[exercise_indices]

        exercise_time[exercise_indices] = t

    # -------------------------------------------------
    # Discount to time 0
    # -------------------------------------------------
    discounted_cashflows = cashflows * np.exp(
        -r * dt * exercise_time
    )

    price = np.mean(discounted_cashflows)

    sd = np.std(discounted_cashflows, ddof=1)

    se = sd / np.sqrt(n_paths)

    ci_95 = (
        price - 1.96 * se,
        price + 1.96 * se
    )

    early_exercise_ratio = np.mean(
        exercise_time < n_steps
    )

    return {
        "n_paths": n_paths,
        "n_steps": n_steps,
        "price": price,
        "sd": sd,
        "se": se,
        "CI_lower": ci_95[0],
        "CI_upper": ci_95[1],
        "early_exercise_ratio": early_exercise_ratio
    }


# =====================================================
# Sensitivity Analysis
# =====================================================

# 想測試的 n_steps
steps_list = [50, 100, 252, 500]

# 想測試的 n_paths
paths_list = [10000, 50000, 100000]

results = []

for n_steps in steps_list:

    for n_paths in paths_list:

        result = price_american_put_lsm(
            n_paths=n_paths,
            n_steps=n_steps
        )

        results.append(result)

# =====================================================
# Convert to DataFrame
# =====================================================

df_results = pd.DataFrame(results)

print(df_results)